# Elastic Net regression

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNet, ElasticNetCV, Lasso, LinearRegression, Ridge, SGDRegressor
from sklearn.model_selection import train_test_split
warnings.simplefilter("ignore")

## Diabetes: four models

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
models = {"Linear regression": LinearRegression(), "Ridge(alpha=0.1)": Ridge(alpha=0.1),
          "Lasso(alpha=0.01)": Lasso(alpha=0.01), "ElasticNet(alpha=0.005, l1_ratio=0.9)": ElasticNet(alpha=0.005, l1_ratio=0.9)}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(f"{name:40s} test R2 {m.score(X_test, y_test):.4f}")

## Choosing alpha and l1_ratio by cross-validation

In [ ]:
cv = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1], alphas=np.logspace(-4, 0, 50), cv=5)
cv.fit(X_train, y_train)
print("best alpha", round(cv.alpha_, 4), " best l1_ratio", cv.l1_ratio_, " test R2", round(cv.score(X_test, y_test), 4))

## The same model with SGDRegressor

In [ ]:
sgd = SGDRegressor(penalty="elasticnet", alpha=0.001, l1_ratio=0.5, max_iter=2000, random_state=0)
sgd.fit(X_train, y_train)
print("SGDRegressor elasticnet test R2", round(sgd.score(X_test, y_test), 4))

## Correlated inputs: the grouping effect

In [ ]:
rng = np.random.default_rng(0)
n = 200
z = rng.standard_normal(n)
X2 = np.c_[z + 0.05 * rng.standard_normal(n), z + 0.05 * rng.standard_normal(n), z + 0.05 * rng.standard_normal(n),
           rng.standard_normal((n, 3))]
y2 = 3 * z + rng.standard_normal(n)
for name, m in [("Linear regression", LinearRegression()), ("Ridge(alpha=10)", Ridge(alpha=10)),
                ("Lasso(alpha=0.1)", Lasso(alpha=0.1)), ("ElasticNet(alpha=0.1, l1_ratio=0.5)", ElasticNet(alpha=0.1, l1_ratio=0.5))]:
    print(f"{name:36s}", m.fit(X2, y2).coef_.round(2))